# Chapter 12 — Object Detection

Maps to Chollet Ch.12. Detection = **boxes + labels**: not just *what* is in the image but *where*.
Output per object: a bounding box `(x, y, w, h)` + a class + a confidence.

### Two families
- **Two-stage (R-CNN / Faster-R-CNN)**: stage 1 proposes regions, stage 2 classifies each. Accurate but slow.
- **Single-stage (YOLO / SSD / RetinaNet)**: predict boxes + classes in **one pass** over a grid. Fast →
  real-time. YOLO is the popular default.

Real detectors train on COCO (18 GB) with a pretrained backbone. To stay runnable, Part 1 builds a
**single-object detector from scratch** on synthetic data (the core ideas: box regression + class head + IoU
+ NMS); the full YOLO-grid / COCO / pretrained RetinaNet paths are in the templates.

In [ ]:
import os; os.environ["KERAS_BACKEND"]="tensorflow"
import keras, numpy as np, matplotlib.pyplot as plt
from keras import layers
from matplotlib.patches import Rectangle


## 1. Synthetic detection data (one object per image)
Each image contains **one** shape: a red **circle** (class 0) or a blue **square** (class 1), at a random
location. Targets: the box `(x, y, w, h)` normalized to [0,1] and the class id.

In [ ]:
H = 64
def make_data(n, seed=0):
    rng = np.random.default_rng(seed)
    X = np.zeros((n, H, H, 3), "float32"); box = np.zeros((n, 4), "float32"); cls = np.zeros((n,), "int32")
    yy, xx = np.mgrid[0:H, 0:H]
    for i in range(n):
        X[i] = rng.uniform(0, 0.15, (H, H, 3))
        k = rng.integers(0, 2)                                  # 0=circle, 1=square
        w = rng.integers(14, 28); h = w if k == 0 else rng.integers(14, 28)
        x0 = rng.integers(0, H-w); y0 = rng.integers(0, H-h)
        if k == 0:
            cy, cx, r = y0+h//2, x0+w//2, w//2
            X[i][(yy-cy)**2 + (xx-cx)**2 <= r*r] = [0.9, 0.2, 0.2]
        else:
            X[i][y0:y0+h, x0:x0+w] = [0.2, 0.4, 0.9]
        box[i] = [x0/H, y0/H, w/H, h/H]; cls[i] = k
    return X, {"box": box, "cls": cls}

Xtr, Ytr = make_data(1200, 0); Xte, Yte = make_data(300, 1)
print("images:", Xtr.shape, " boxes:", Ytr["box"].shape, " classes:", np.unique(Ytr["cls"]))


## 2. A detector = backbone + two heads
A ConvNet backbone extracts features, then **two heads** branch off (Functional API, Ch.7):
- **box head**: `Dense(4, "sigmoid")` → `(x, y, w, h)` in [0,1] (regression, MSE loss).
- **class head**: `Dense(C, "softmax")` → class (crossentropy loss).

> **Key detail:** use **`Flatten`**, not `GlobalAveragePooling`, before the heads. GAP averages away *where*
> things are — fine for classification, fatal for localization (same lesson as segmentation in Ch.11).
> The combined loss is a weighted sum; box loss gets a higher weight so coordinates are learned precisely.

In [ ]:
inputs = keras.Input((H, H, 3))
x = layers.Conv2D(32, 3, strides=2, activation="relu", padding="same")(inputs)   # 32
x = layers.Conv2D(64, 3, strides=2, activation="relu", padding="same")(x)        # 16
x = layers.Conv2D(128, 3, strides=2, activation="relu", padding="same")(x)       # 8
x = layers.Flatten()(x)                                  # keep spatial info for localization!
x = layers.Dense(128, activation="relu")(x); x = layers.Dropout(0.3)(x)
box_head = layers.Dense(4, activation="sigmoid", name="box")(x)
cls_head = layers.Dense(2, activation="softmax", name="cls")(x)
detector = keras.Model(inputs, {"box": box_head, "cls": cls_head})

detector.compile(
    optimizer="adam",
    loss={"box": "mse", "cls": "sparse_categorical_crossentropy"},
    loss_weights={"box": 50.0, "cls": 1.0},              # weight box higher so coords are precise
    metrics={"cls": "accuracy"},
)
detector.fit(Xtr, Ytr, epochs=25, batch_size=32, validation_split=0.1, verbose=0)
print("trained.")


## 3. Evaluate with IoU (Intersection over Union)
A predicted box "hits" if its IoU with the true box exceeds a threshold (e.g. 0.5). IoU = overlap area ÷
union area.

In [ ]:
def iou(a, b):
    ax, ay, aw, ah = a; bx, by, bw, bh = b
    ix = max(0, min(ax+aw, bx+bw) - max(ax, bx))
    iy = max(0, min(ay+ah, by+bh) - max(ay, by))
    inter = ix*iy; union = aw*ah + bw*bh - inter
    return inter/union if union > 0 else 0.0

pred = detector.predict(Xte, verbose=0)
ious = [iou(pred["box"][i], Yte["box"][i]) for i in range(len(Xte))]
cls_acc = (pred["cls"].argmax(1) == Yte["cls"]).mean()
hits = np.mean([v > 0.5 for v in ious])
print(f"mean IoU = {np.mean(ious):.3f}   class acc = {cls_acc:.3f}   detection@IoU>0.5 = {hits:.3f}")


## 4. Visualize predictions (green = true, red = predicted)

In [ ]:
CLASSES = ["circle", "square"]
def show_detection(idx):
    img = Xte[idx]; p = detector.predict(img[None], verbose=0)
    pb = p["box"][0]*H; tb = Yte["box"][idx]*H
    pc = CLASSES[p["cls"][0].argmax()]
    fig, ax = plt.subplots(figsize=(3,3)); ax.imshow(img); ax.axis("off")
    ax.add_patch(Rectangle((tb[0],tb[1]), tb[2],tb[3], ec="lime", fc="none", lw=2))
    ax.add_patch(Rectangle((pb[0],pb[1]), pb[2],pb[3], ec="red",  fc="none", lw=2, ls="--"))
    ax.set_title(f"pred={pc}  IoU={iou(p['box'][0],Yte['box'][idx]):.2f}")
    plt.show()

for idx in [0, 1, 2]:
    show_detection(idx)


## 5. Non-Max Suppression (NMS) — clean up duplicate boxes
A real detector outputs many overlapping boxes for one object. **NMS** keeps the highest-confidence box and
removes others that overlap it above an IoU threshold. Essential post-processing for any multi-box detector.

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    order = list(np.argsort(scores)[::-1])    # highest score first
    keep = []
    while order:
        i = order.pop(0); keep.append(i)
        order = [j for j in order if iou(boxes[i], boxes[j]) < iou_threshold]
    return keep

boxes  = np.array([[0,0,.5,.5], [0.03,0.03,.5,.5], [0.6,0.6,.3,.3]])   # first two overlap
scores = np.array([0.95, 0.80, 0.70])
print("kept boxes after NMS:", nms(boxes, scores, 0.5), " (dropped the duplicate)")


### How real single-stage detectors (YOLO) scale this up
- Divide the image into an **S×S grid**; each cell predicts box(es) + confidence + class probabilities **in one
  pass** (no region proposals).
- A cell is "responsible" for objects whose **center** falls in it.
- Train with a combined localization + confidence + classification loss; at inference, threshold confidence
  then apply **NMS**.
- Quality is summarized by **mAP** (mean Average Precision) across classes and IoU thresholds.

---
# ✍️ PROBLEMS

### P1 — GAP vs Flatten ablation
Rebuild the detector with `GlobalAveragePooling2D` instead of `Flatten`. Compare mean IoU. Confirm GAP
destroys localization (much lower IoU) while class accuracy stays high. Explain why.

In [ ]:
# TODO


### P2 — mean IoU vs box-loss weight
Sweep `loss_weights["box"] ∈ {1, 10, 50, 200}`. Plot mean test IoU vs weight. Is there a sweet spot? What
happens to class accuracy at extreme weights?

In [ ]:
# TODO


### P3 — Add a confidence output
Add a third head `Dense(1, "sigmoid")` predicting "object present" and train with some **empty** images
(no object, box=0, confidence target 0). At test, suppress boxes with confidence < 0.5.

In [ ]:
# TODO


### P4 — Two objects + NMS
Extend `make_data` to place TWO shapes per image and switch to a small **grid** head (2×2) predicting a box
per cell. Decode predictions, run your `nms`, and visualize. (This is a mini-YOLO.)

In [ ]:
# TODO


---
# 📋 TEMPLATES

### T1 — Detector heads (box regression + classification)

In [ ]:
import keras
from keras import layers
inputs = keras.Input((H, W, 3))
x = backbone(inputs)                 # any ConvNet feature extractor
x = layers.Flatten()(x)              # Flatten (NOT GAP) to keep location for boxes
x = layers.Dense(256, activation="relu")(x)
box = layers.Dense(4, activation="sigmoid", name="box")(x)     # x,y,w,h in [0,1]
cls = layers.Dense(NUM_CLASSES, activation="softmax", name="cls")(x)
model = keras.Model(inputs, {"box": box, "cls": cls})
model.compile("adam",
              loss={"box": "mse", "cls": "sparse_categorical_crossentropy"},
              loss_weights={"box": 50.0, "cls": 1.0}, metrics={"cls": "accuracy"})


### T2 — IoU and NMS

In [ ]:
def iou(a, b):
    ax,ay,aw,ah=a; bx,by,bw,bh=b
    ix=max(0,min(ax+aw,bx+bw)-max(ax,bx)); iy=max(0,min(ay+ah,by+bh)-max(ay,by))
    inter=ix*iy; u=aw*ah+bw*bh-inter; return inter/u if u>0 else 0.0
def nms(boxes, scores, thr=0.5):
    order=list(np.argsort(scores)[::-1]); keep=[]
    while order:
        i=order.pop(0); keep.append(i)
        order=[j for j in order if iou(boxes[i],boxes[j])<thr]
    return keep


### T3 — YOLO-style grid head (single-stage, Colab with a real backbone)

In [ ]:
# grid_size, num_labels given. backbone from keras_hub (e.g. resnet_50_imagenet)
# x = backbone(inputs)
# x = layers.Conv2D(512, 3, strides=2)(x); x = layers.Flatten()(x)
# x = layers.Dense(2048, activation="relu")(x); x = layers.Dropout(0.5)(x)
# x = layers.Dense(grid_size*grid_size*(num_labels+5))(x)
# x = layers.Reshape((grid_size, grid_size, num_labels+5))(x)
# box = x[..., :5]                      # (x,y,w,h,confidence) per cell
# cls = layers.Activation("softmax")(x[..., 5:])
# outputs = {"box": box, "class": cls}


### T4 — Pretrained detector (zero training, Colab)

In [ ]:
# import keras_hub
# detector = keras_hub.models.ImageObjectDetector.from_preset("retinanet_resnet50_fpn_coco")
# preds = detector.predict(images)     # boxes + classes + confidences; then threshold + NMS


---
### ✅ Checklist
- [ ] Explain detection vs segmentation/classification and two-stage vs single-stage.
- [ ] Build a detector with separate box (regression) + class (softmax) heads; use Flatten not GAP.
- [ ] Implement IoU; evaluate detection@IoU>0.5.
- [ ] Implement NMS and explain why it's needed.
- [ ] Know the YOLO grid idea, confidence scores, and that mAP is the standard metric.

**Next: Chapter 13** — *Timeseries forecasting* (windows, baselines, RNN/LSTM for sequences). Say "Chapter 13".